# MEGASWEEP

### Imports

In [ ]:
%%capture
%pip install numpy pandas statsmodels matplotlib itertools arch joblib

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import itertools
import warnings
from arch import arch_model
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
np.random.seed(42)

# Global Configuration
N = 5000         # Total length of each path
M = 5            # Number of paths per scenario (reduced for speed)
test_window = 50 # How many steps to backtest
train_size = 500 # Size of rolling training window

### Scoring

In [ ]:
def winkler(y, m, v, a=0.05):
    """Calculates the Winkler Score for interval forecasting."""
    d = 1.96 * np.sqrt(v)
    l, u = m - d, m + d
    return (u - l) + (2/a) * max(l - y, 0) + (2/a) * max(y - u, 0)

def analyze(y, p, v):
    """Calculates performance metrics for a prediction series."""
    y, p, v = np.array(y), np.array(p), np.array(v)
    s = np.sqrt(v)
    rets = np.sign(p) * y
    
    return [
        np.mean((y - p)**2),                            # MSE
        np.mean([winkler(yi, pi, vi) for yi, pi, vi in zip(y, p, v)]), # Winkler
        np.mean((y >= p - 1.96*s) & (y <= p + 1.96*s)), # Cov95
        np.mean(np.sign(y) == np.sign(p)),               # HitRate
        (np.mean(rets) / np.std(rets)) * np.sqrt(252) if np.std(rets) > 0 else 0, # Sharpe
        np.sum(rets)                                    # TotRet
    ]

### Processing

In [ ]:
def process_step(tr, act, sweeps, is_last_step=False):
    step_results = {}
    failed_models = []
    step_params = {} # NEW: Dict to hold parameters
    
    for name, (m_type, val) in sweeps.items():
        try:
            if m_type == "ar":
                m = sm.tsa.AutoReg(tr, val).fit()
                step_results[name] = [m.predict(len(tr), len(tr))[0], m.sigma2]
                if is_last_step: step_params[name] = m.params.values
            
            elif m_type in ["arch", "garch"]:
                p, q = (val, 0) if m_type == "arch" else val
                m = arch_model(tr, vol='Garch', p=p, q=q).fit(disp='off')
                f = m.forecast(horizon=1)
                step_results[name] = [f.mean.values[-1,0], f.variance.values[-1,0]]
                if is_last_step: step_params[name] = m.params.values
            
            elif m_type == "ms":
                m = sm.tsa.MarkovAutoregression(tr, k_regimes=2, order=val, 
                                                switching_variance=True).fit(search_reps=20, disp=False)
                p_nxt = np.asarray(m.regime_transition).reshape(2,2) @ \
                        np.asarray(m.filtered_marginal_probabilities)[-1].flatten()
                lags = tr[-val:][::-1]
                mu_regimes = [m.params[2+r] + np.dot(m.params[4+r:4+2*val:2], lags) for r in range(2)]
                step_results[name] = [np.dot(p_nxt, mu_regimes), np.dot(p_nxt, m.params[-2:])]
                if is_last_step: step_params[name] = m.params.values
                
        except Exception:
            failed_models.append(name)
            
    return step_results, failed_models, step_params

### Sweep grid

In [ ]:
# Transition Matrices
P_grids = [
    [[0.98, 0.02], [0.05, 0.95]], # Stable Regimes
    [[0.70, 0.30], [0.30, 0.70]]  # Churning/Noisy Regimes
]

# Regime Means
mu_grids = [
    [0.0, 0.0],   # No drift
    [-1.0, 1.0]   # Structural breaks in mean
]

# Regime Volatilities
sig_grids = [
    [0.1, 1.5],   # High variance contrast
    [1.0, 1.0]    # Constant variance (Homoskedastic)
]

phi = [0.9, 0.2] # Autoregressive persistence
scenarios = list(itertools.product(P_grids, mu_grids, sig_grids))

sweeps = {
    "AR(1)": ("ar", 1), 
    "GARCH(1,1)": ("garch", (1,1)), 
    "MSAR-O1": ("ms", 1) 
}

print(f"Grid defined. Total scenarios to test: {len(scenarios)}")

### Master execution

In [ ]:
master_results = {}

for s_idx, (P, mu, sig) in enumerate(scenarios):
    s_name = f"S{s_idx}: P{P[0][0]}_M{mu[0]}_S{sig[0]}"
    print(f"\nProcessing {s_name}...")
    
    paths = []
    for _ in range(M):
        s, y = [0], [0.]
        for t in range(1, N):
            s.append(np.random.choice([0, 1], p=P[s[-1]]))
            y.append(mu[s[-1]] + phi[s[-1]] * y[-1] + np.random.normal(0, sig[s[-1]]))
        paths.append(np.array(y))
    
    all_results = {k: [] for k in sweeps}
    scenario_params = {k: [] for k in sweeps} # NEW: Track params for this scenario
    
    for p_idx, path in enumerate(paths):
        results = Parallel(n_jobs=-1)(
            # NEW: pass i == 1 to flag the last step
            delayed(process_step)(path[-i-train_size:-i], path[-i], sweeps, i == 1) 
            for i in range(test_window, 0, -1)
        )
        
        path_hist = {k: [] for k in sweeps}
        for step_res, fails, params in results:
            for name in sweeps:
                if name in step_res: path_hist[name].append(step_res[name])
                # NEW: If params were returned for this model, save them
                if name in params: scenario_params[name].append(params[name])
        
        for k in sweeps: all_results[k].append(path_hist[k])
    
    metrics = {}
    for mod in all_results:
        m_list = [analyze(paths[i][-test_window:], np.array(pv)[:,0], np.array(pv)[:,1]) 
                  for i, pv in enumerate(all_results[mod]) if len(pv) == test_window]
        if m_list: metrics[mod] = np.mean(m_list, axis=0)
    
    # NEW: Average the recovered params across all paths and store them
    avg_params = {mod: np.mean(scenario_params[mod], axis=0) for mod in sweeps if scenario_params[mod]}
    
    # Store both metrics and parameters
    master_results[s_name] = {
        "metrics": metrics,
        "true_dgp": {"P": P, "mu": mu, "sig": sig, "phi": phi},
        "est_params": avg_params
    }

### Analysis

In [ ]:
headers = ["MSE", "Winkler", "Cov95", "HitRate", "Sharpe", "TotRet"]

for s_name, data in master_results.items():
    print(f"\n--- {s_name} ---")
    print(f"{'MODEL':<12} | " + " | ".join(f"{col:<7}" for col in headers))
    print("-" * 75)
    for mod, vals in data.items():
        print(f"{mod:<12} | " + " | ".join(f"{v:<7.3f}" for v in vals))